# Fitted — Recommender Playground

Play with the recommendation pipeline end-to-end, locally.  
Requires: DB accessible via `DATABASE_URL` (direct or SSH tunnel) and CLIP weights cached locally.

**Sections**
1. Setup
2. Embedding service — encode text queries with CLIP
3. Dev catalog search — ANN search on `catalog_items`
4. User embedding — build a user vector from style prefs
5. Two-tower ranking — score items against a user vector
6. Full pipeline — `RecommendationService.recommend()`
7. Interactive exploration — compare queries, tweak top-k

## 1. Setup

In [ ]:
import sys, os

# Add project root to path so `app.*` imports work
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
# Async support in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    print("Install nest_asyncio:  uv pip install nest_asyncio")
    raise

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
)
# Quieten noisy deps
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("psycopg").setLevel(logging.WARNING)
logging.getLogger("psycopg_pool").setLevel(logging.WARNING)

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
# Set DATABASE_URL if not already in the environment.
# For local dev: direct connection string.
# For EC2/RDS: open an SSH tunnel first:
#   ssh -L 5432:<rds-endpoint>:5432 ec2-user@<ec2-ip> -N &
# then use:  postgresql://fitted:<password>@localhost:5432/fitted

if not os.environ.get("DATABASE_URL"):
    os.environ["DATABASE_URL"] = input("DATABASE_URL: ").strip()

# DEV_MODE bypasses JWT auth on the backend — useful for testing
os.environ.setdefault("DEV_MODE", "true")

print("DATABASE_URL:", os.environ["DATABASE_URL"][:40], "...")

## 2. Embedding Service — encode text with CLIP

First call downloads / loads CLIP ViT-B/32 weights (~600 MB). Subsequent calls use the module-level singleton.

In [ ]:
from app.services.embedding_service import encode_text
import numpy as np

queries = [
    "navy blue blazer men",
    "casual streetwear hoodie",
    "floral summer dress",
    "minimalist white sneakers",
]

embeddings = {q: encode_text(q) for q in queries}

for q, emb in embeddings.items():
    print(f"{q!r:40s}  shape={emb.shape}  norm={np.linalg.norm(emb):.4f}")

In [ ]:
# Pairwise cosine similarities (all unit vectors → dot product = cosine)
import itertools

print("Cosine similarities between query pairs:")
for a, b in itertools.combinations(queries, 2):
    sim = float(np.dot(embeddings[a], embeddings[b]))
    print(f"  {a!r:35s} ↔ {b!r:35s}  {sim:+.4f}")

## 3. Database — init pool and catalog search

In [ ]:
import asyncio
from app.services import db_service

await db_service.init_pool()
print("DB pool initialized")

In [ ]:
# Quick sanity check — count catalog items
from app.services.db_service import get_connection

async with get_connection() as conn:
    async with conn.cursor() as cur:
        await cur.execute(
            "SELECT COUNT(*), COUNT(embedding) FROM catalog_items WHERE domain = 'fashion'"
        )
        total, embedded = await cur.fetchone()

print(f"catalog_items (fashion):  {total} total,  {embedded} embedded")

In [ ]:
# Browse raw catalog rows
from app.services.db_service import get_connection

async with get_connection() as conn:
    async with conn.cursor() as cur:
        await cur.execute(
            "SELECT item_id, title, price, source, attributes FROM catalog_items WHERE domain = 'fashion' LIMIT 10"
        )
        rows = await cur.fetchall()

for item_id, title, price, source, attributes in rows:
    print(f"[{item_id[:8]}]  ${price:6.2f}  {source:10s}  {title[:55]}")

In [ ]:
# ANN search — find nearest neighbors for a query
from app.services.dev_catalog_service import search as catalog_search

QUERY = "navy blue blazer men"  # ← change me
query_emb = encode_text(QUERY)

candidates = await catalog_search(query_emb, limit=10)

print(f"Query: {QUERY!r}  → {len(candidates)} candidates")
print()
for i, item in enumerate(candidates, 1):
    print(f"  {i:2d}. [{item.item_id[:8]}]  {item.title[:55]:55s}  ${item.price:.2f}")

In [ ]:
# Inspect a single item in detail
item = candidates[0]
print("item_id  :", item.item_id)
print("title    :", item.title)
print("price    :", item.price)
print("source   :", item.source)
print("image    :", item.image_url)
print("url      :", item.product_url)
print("embedding:", None if item.embedding is None else f"shape={item.embedding.shape}  norm={np.linalg.norm(item.embedding):.4f}")
print("attributes:", item.attributes)

## 4. User Embedding

Build a user vector from style preferences (cold-start path — no wardrobe items needed).

In [ ]:
from unittest.mock import MagicMock, patch
from app.services.recommendation_service import RecommendationService, UserTower, ItemTower

# Build service without hitting S3 for tower weights
with patch("app.services.recommendation_service._load_towers_from_s3", return_value=None):
    svc = RecommendationService(s3_client=MagicMock(), bucket="local")

print("UserTower W shape:", svc.user_tower.W.shape)
print("ItemTower W shape:", svc.item_tower.W.shape)

In [ ]:
# Build a user embedding from style preferences
# (Uses the cold-start path: encode each tag, mean-pool, project through UserTower)

FAKE_USER_ID = "00000000-0000-0000-0000-000000000001"

STYLE_PREFS = {
    "styles": ["streetwear", "casual"],
    "colors": ["black", "grey"],
}

# Patch get_connection to return an empty wardrobe (cold-start scenario)
from contextlib import asynccontextmanager
from unittest.mock import AsyncMock

@asynccontextmanager
async def _empty_conn():
    mock_cur = AsyncMock()
    mock_cur.fetchall = AsyncMock(return_value=[])  # no wardrobe items
    mock_cur_ctx = MagicMock()
    mock_cur_ctx.__aenter__ = AsyncMock(return_value=mock_cur)
    mock_cur_ctx.__aexit__ = AsyncMock(return_value=False)
    mock_conn = MagicMock()
    mock_conn.cursor = MagicMock(return_value=mock_cur_ctx)
    yield mock_conn

with patch("app.services.recommendation_service.get_connection", return_value=_empty_conn()):
    user_emb = await svc._build_user_embedding(FAKE_USER_ID, STYLE_PREFS)

print("User embedding shape:", user_emb.shape)
print("Norm:", np.linalg.norm(user_emb))

In [ ]:
# Or — if you have a real user in the DB — use the real wardrobe path
# Uncomment and set a real user_id:

# REAL_USER_ID = "<uuid-from-users-table>"
# real_user_emb = await svc._build_user_embedding(REAL_USER_ID, STYLE_PREFS)
# print("Real user embedding norm:", np.linalg.norm(real_user_emb))

## 5. Two-Tower Ranking

Score the ANN candidates against the user embedding using the two towers.

In [ ]:
# Rank the candidates we retrieved in section 3
# (candidates may have embeddings from the DB, or encode_text is called on-the-fly)

ranked = svc.rank(user_emb, candidates)

print(f"Ranked {len(ranked)} items:")
print()
for i, (item, score) in enumerate(ranked, 1):
    print(f"  {i:2d}.  score={score:+.4f}  {item.title[:55]:55s}  ${item.price:.2f}")

In [ ]:
# Compare ranking with identity towers (no projection distortion)
# Useful for understanding the raw CLIP similarity ordering vs two-tower ordering

identity = np.eye(512, dtype=np.float32)
svc_identity = RecommendationService.__new__(RecommendationService)
svc_identity.user_tower = UserTower(weights=identity)
svc_identity.item_tower = ItemTower(weights=identity)

from app.services.domain_factory import get_domain
svc_identity._domain = get_domain("fashion")

raw_query_emb = encode_text(QUERY)  # raw CLIP query (not user-projected)
ranked_identity = svc_identity.rank(raw_query_emb, candidates)

print("Raw CLIP cosine ordering (identity towers):")
for i, (item, score) in enumerate(ranked_identity, 1):
    print(f"  {i:2d}.  cosine={score:+.4f}  {item.title[:55]}")

## 6. Full Pipeline — `RecommendationService.recommend()`

End-to-end: LLM search query → CLIP embed → vector cache → ANN → rank → ProductRecommendation list.

> Requires `OPENROUTER_API_KEY` for the LLM step. If missing, the service falls back to a rule-based query.

In [ ]:
# Set OpenRouter key if you have one (optional — rule-based fallback works fine)
if not os.environ.get("OPENROUTER_API_KEY"):
    key = input("OPENROUTER_API_KEY (leave blank to skip LLM): ").strip()
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
    else:
        print("Skipping LLM — will use rule-based query generation fallback")

In [ ]:
# Run the full pipeline
# vector_cache.lookup/store are skipped here (no S3 configured locally)
# — we patch them out so the ANN path always runs

from unittest.mock import patch, AsyncMock

WEATHER_CTX = {
    "temp_c": 18.0,
    "condition": "Partly cloudy",
    "feels_like_c": 16.5,
}

STYLE_PREFS = {
    "styles": ["streetwear", "casual"],
    "colors": ["black", "navy"],
}

with patch("app.services.vector_cache.lookup", new=AsyncMock(return_value=None)), \
     patch("app.services.vector_cache.store",  new=AsyncMock(return_value="local-cache")), \
     patch("app.services.recommendation_service.get_connection", return_value=_empty_conn()):

    results = await svc.recommend(
        user_id=FAKE_USER_ID,
        location="London",
        weather_context=WEATHER_CTX,
        style_preferences=STYLE_PREFS,
        top_k=10,
        include_explanation=False,
    )

print(f"Got {len(results)} recommendations:")
print()

In [ ]:
# Display results as a table
import json

for i, rec in enumerate(results, 1):
    attrs = ", ".join(f"{k}={v}" for k, v in (rec.attributes or {}).items() if k in ("brand", "size", "condition"))
    print(f"{i:2d}. [{rec.similarity_score:+.4f}]  {rec.title[:50]:50s}  ${rec.price:.2f}")
    if attrs:
        print(f"         {attrs}")
    print(f"         {rec.product_url}")

In [ ]:
# With LLM explanation (requires OPENROUTER_API_KEY)
if os.environ.get("OPENROUTER_API_KEY"):
    with patch("app.services.vector_cache.lookup", new=AsyncMock(return_value=None)), \
         patch("app.services.vector_cache.store",  new=AsyncMock(return_value="local-cache")), \
         patch("app.services.recommendation_service.get_connection", return_value=_empty_conn()):

        results_with_exp = await svc.recommend(
            user_id=FAKE_USER_ID,
            location="London",
            weather_context=WEATHER_CTX,
            style_preferences=STYLE_PREFS,
            top_k=5,
            include_explanation=True,
        )

    print("Explanation:")
    print(results_with_exp[0].llm_explanation if results_with_exp else "(no results)")
else:
    print("Skipped — no OPENROUTER_API_KEY")

## 7. Interactive Exploration

Tweak queries, style prefs, and temperatures to see how the pipeline responds.

In [ ]:
# ── Compare two queries side-by-side ─────────────────────────────────────────

async def search_and_rank(query: str, style_prefs: dict, top_k: int = 5):
    """Embed a query, search the catalog, rank results, return top-k."""
    q_emb = encode_text(query)
    items = await catalog_search(q_emb, limit=50)
    if not items:
        return []

    # Build user embedding from style prefs (cold-start)
    style_tags = style_prefs.get("styles", []) + style_prefs.get("colors", [])
    if style_tags:
        vecs = np.stack([encode_text(t) for t in style_tags])
        u_raw = vecs.mean(0)
        u_raw /= np.linalg.norm(u_raw)
    else:
        u_raw = q_emb

    u_emb = svc.user_tower.forward(u_raw)
    return svc.rank(u_emb, items)[:top_k]


QUERY_A = "slim fit chinos beige"
QUERY_B = "oversized graphic tee"
PREFS   = {"styles": ["casual"], "colors": ["neutral"]}

results_a = await search_and_rank(QUERY_A, PREFS)
results_b = await search_and_rank(QUERY_B, PREFS)

print(f"=== {QUERY_A!r} ===")
for item, score in results_a:
    print(f"  {score:+.4f}  {item.title[:55]}")

print()
print(f"=== {QUERY_B!r} ===")
for item, score in results_b:
    print(f"  {score:+.4f}  {item.title[:55]}")

In [ ]:
# ── Score distribution for a single query ────────────────────────────────────

QUERY = "leather jacket biker"  # ← change me

q_emb = encode_text(QUERY)
items = await catalog_search(q_emb, limit=100)
ranked_all = svc.rank(user_emb, items)

scores = [s for _, s in ranked_all]
print(f"Query: {QUERY!r}  —  {len(scores)} items")
print(f"  min={min(scores):+.4f}  max={max(scores):+.4f}  mean={np.mean(scores):+.4f}  std={np.std(scores):.4f}")

# Histogram (ASCII)
bins = np.linspace(min(scores), max(scores), 11)
hist, _ = np.histogram(scores, bins=bins)
print()
for lo, hi, count in zip(bins, bins[1:], hist):
    bar = "█" * int(count * 40 / max(hist, 1))
    print(f"  [{lo:+.3f}, {hi:+.3f})  {bar} {count}")

In [ ]:
# ── Effect of identity vs Xavier towers on ranking ───────────────────────────

QUERY = "white linen shirt summer"  # ← change me

q_emb   = encode_text(QUERY)
items   = await catalog_search(q_emb, limit=20)

# Xavier-initialised ranking (current production default)
xavier_ranked = svc.rank(user_emb, items)[:5]

# Identity-tower ranking (raw CLIP cosine)
identity_ranked = svc_identity.rank(q_emb, items)[:5]

print("Xavier towers (user-projected):")
for i, (item, s) in enumerate(xavier_ranked, 1):
    print(f"  {i}. {s:+.4f}  {item.title[:50]}")

print()
print("Identity towers (raw CLIP cosine):")
for i, (item, s) in enumerate(identity_ranked, 1):
    print(f"  {i}. {s:+.4f}  {item.title[:50]}")

In [ ]:
# ── Cleanup ──────────────────────────────────────────────────────────────────
await db_service.close_pool()
print("DB pool closed")